<a href="https://colab.research.google.com/github/MadhawaRathnayake/DataPreprocessing-HPC/blob/cuda_analyzer/modules/analyzer_cuda/cuda_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MadhawaRathnayake/DataPreprocessing-HPC.git
%cd DataPreprocessing-HPC
!git branch -a


Cloning into 'DataPreprocessing-HPC'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 183 (delta 97), reused 181 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 2.60 MiB | 14.61 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/DataPreprocessing-HPC
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/Pipeline-Wired-up
  remotes/origin/cuda_analyzer
  remotes/origin/main
  remotes/origin/series-openmp-mpi-integration


In [ ]:
!git checkout -b cuda_analyzer origin/cuda_analyzer


Branch 'cuda_analyzer' set up to track remote branch 'cuda_analyzer' from 'origin'.
Switched to a new branch 'cuda_analyzer'


In [ ]:
!git branch
!git status

* cuda_analyzer
  main
On branch cuda_analyzer
Your branch is up to date with 'origin/cuda_analyzer'.

nothing to commit, working tree clean


In [ ]:
!git branch
!pwd
!ls

* cuda_analyzer
  main
/content/DataPreprocessing-HPC
build.sh  MODULAR_UI_CHANGES.md  QUICKSTART.md	SKILL.md
data	  modules		 README.md	ui
lib	  PROJECT_OVERVIEW.md	 run.sh


In [ ]:
%cd /content/DataPreprocessing-HPC/modules
!ls

/content/DataPreprocessing-HPC/modules
analyzer_cuda  analyzer_mpi  analyzer_openmp  importer	series_processing


In [ ]:
%cd /content/DataPreprocessing-HPC/modules/analyzer_cuda
!ls

/content/DataPreprocessing-HPC/modules/analyzer_cuda
cuda_analyzer.ipynb


In [ ]:
import os
os.chdir('/content/DataPreprocessing-HPC/modules/analyzer_cuda')
print(os.getcwd())

/content/DataPreprocessing-HPC/modules/analyzer_cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/DataPreprocessing-HPC/modules/analyzer_cuda

/content/DataPreprocessing-HPC/modules/analyzer_cuda


In [ ]:
!find /content/DataPreprocessing-HPC -name "cuda_analyzer.ipynb"

/content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb


In [ ]:
import os
os.chdir('/content/DataPreprocessing-HPC/modules/analyzer_cuda')
print(os.getcwd())
!ls -la

/content/DataPreprocessing-HPC/modules/analyzer_cuda
total 8
drwxr-xr-x 2 root root 4096 Mar  9 00:22 .
drwxr-xr-x 7 root root 4096 Mar  9 00:22 ..
-rw-r--r-- 1 root root    0 Mar  9 00:22 cuda_analyzer.ipynb


In [ ]:
import json

nb = {
    "cells": [
        {
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": [
                "# Start coding here\n",
                "print('Hello from cuda_analyzer')\n"
            ]
        }
    ],
    "metadata": {
        "colab": {"name": "cuda_analyzer.ipynb"},
        "kernelspec": {
            "display_name": "Python 3",
            "name": "python3"
        },
        "language_info": {"name": "python"}
    },
    "nbformat": 4,
    "nbformat_minor": 0
}

path = "/content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb"
with open(path, "w") as f:
    json.dump(nb, f)

print("Created valid notebook at:", path)

Created valid notebook at: /content/DataPreprocessing-HPC/modules/analyzer_cuda/cuda_analyzer.ipynb


In [1]:
!git branch

fatal: not a git repository (or any of the parent directories): .git


In [2]:
import os
print(os.getcwd())

/content


In [3]:
%cd /content/DataPreprocessing-HPC
!git branch
!git status

[Errno 2] No such file or directory: '/content/DataPreprocessing-HPC'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
%%writefile /content/vector_add.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A, const float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    const int N = 8;
    size_t size = N * sizeof(float);

    float h_A[N] = {1, 2, 3, 4, 5, 6, 7, 8};
    float h_B[N] = {10, 20, 30, 40, 50, 60, 70, 80};
    float h_C[N];

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    printf("Result:\\n");
    for (int i = 0; i < N; i++) {
        printf("%.1f ", h_C[i]);
    }
    printf("\\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}